# Brazil and Chile: original financial reports

Install OpenCorpFilings with the `pdf` extra. These two experimental routes
need no credentials. Each download below selects one issuer and one annual report.
Outputs were captured in October 2026; later runs can select newer reports.

Brazil reads CVM IPE PDFs, excluding the structured DFP/ITR datasets. Chile
reads the CMF securities-issuer IFRS portal. Both preserve original URLs and
source metadata. Attribute CVM/CMF; the library's MIT license covers code,
not the issuer reports. See the market guides for source-access terms.


In [1]:
from pathlib import Path
from opencorpfilings import Company, search_companies

output = Path("downloads/brazil-chile")
output.mkdir(parents=True, exist_ok=True)


## Brazil: search a regulator name, then select annual statements

A CVM code or CNPJ also works directly. The data host requires ten-second
pacing; the first lookup reads its company registry and delivery-year metadata.
The combined `find_company` index also includes active Brazil and Chile issuer pools.


In [2]:
brazil_matches = search_companies("Petrobras", market="BR", limit=1)
petrobras = Company(brazil_matches[0])
br_report = petrobras.filings(form="annual", limit=1).latest()
print(petrobras.entity.name)
print(br_report)
br_report.primary_document.download(output / "petrobras-annual.pdf")
print(br_report.primary_document.page(1).text[:300])


PETRÓLEO BRASILEIRO S.A. - PETROBRAS
<Filing 2026:009512IPE311220250151708258-10:1 Demonstrações Financeiras Anuais Completas ANNUAL period=2025-12-31 filed=2026-03-05 BR/brazil_cvm>
 
 
 
 
 
 
 
 
RELATÓRIO DA             
Administração 
2025 
 
 
 
 
 
 
 



## Chile: a report period is not a filing date

CMF's displayed consultation time is the query time. The route leaves
`filing_date` and `published_at` unset and sorts by verified report period.
Consolidated reports are the default; `consolidated=False` selects individual
reports where available. Source-supplied signed file links can expire: open
a fresh company and select the report again if needed.


In [3]:
chile_matches = search_companies("Copec", market="CL", limit=1)
copec = Company(chile_matches[0])
cl_report = copec.filings(form="annual", limit=1).latest()
print(copec.entity.name)
print(cl_report)
print("Report period:", cl_report.period_end)
print("Filing date:", cl_report.filing_date)
cl_report.primary_document.download(output / "copec-annual.pdf")
print(cl_report.primary_document.page(1).text.strip()[:300])


EMPRESAS COPEC S.A.
<Filing 90690000:202512:C IFRS CONSOLIDADO ANNUAL period=2025-12-31 CL/chile_cmf>
Report period: 2025-12-31
Filing date: None
i 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
ESTADOS FINANCIEROS  
CONSOLIDADOS  
al 31 de diciembre de 2025


## Read the original XBRL occurrences

The source package includes a classic `.xbrl` instance. These are filed values,
contexts and units, with source locations; they are not standardized statements.
Brazil's PDF-only IPE route does not claim raw facts.


In [4]:
facts = cl_report.facts
print("Raw fact occurrences:", len(facts))
for fact in facts[:3]:
    print(fact.concept_raw, fact.value, fact.unit)
petrobras.close()
copec.close()


Raw fact occurrences: 10274
ifrs-full:CashAndCashEquivalents 2461976000 iso4217:USD
ifrs-full:OtherCurrentFinancialAssets 314559000 iso4217:USD
ifrs-full:OtherCurrentNonfinancialAssets 282130000 iso4217:USD
